In [1]:
import pandas as pd
import numpy as np
import time

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from reporting import *

In [2]:
# Define dtype map (force string types for mixed-type columns)
dtype_map = {
    6: str,  # Income Group ID
    7: str,  # Income Group Name
    12: str,  # Managing Sub-agency or Bureau ID
}

# Load CSV with dtype enforcement for non-integer columns, handling potential parsing issues
df_complete = pd.read_csv('downloads/us_foreign_aid_complete.csv', dtype=dtype_map, low_memory=False)

# Clean column 48 (Fiscal Year) by extracting valid 4-digit years
df_complete.iloc[:, 48] = pd.to_numeric(
    df_complete.iloc[:, 48].astype(str).str.extract(r'(\d{4})')[0],
    errors='coerce'
).astype('Int64')  # Use nullable integer type

# Handle missing or invalid values
print(f"Column 48 (Fiscal Year): {df_complete.iloc[:, 48].isna().sum()} rows had invalid year values and were set to NaN.")

# Convert other numeric columns safely (if needed, add more columns here)
numeric_columns = [48]  # Add other numeric columns if required
for col in numeric_columns:
    df_complete.iloc[:, col] = pd.to_numeric(df_complete.iloc[:, col], errors='coerce').astype('Int64')


Column 48 (Fiscal Year): 0 rows had invalid year values and were set to NaN.


In [28]:
df_complete[df_complete["Country Code"] == "MMR"]

,Country ID,Country Code,Country Name,Region ID,Region Name,Income Group ID,Income Group Name,Income Group Acronym,Managing Agency ID,Managing Agency Acronym,...,Transaction Type ID,Transaction Type Name,Fiscal Year,Transaction Date,Current Dollar Amount,Constant Dollar Amount,aid_type_id,aid_type_name,activity_budget_amount,submission_activity_id
268942,104,MMR,Burma (Myanmar),1,East Asia and Oceania,2.0,Lower Middle Income Country,LMIC,1,USAID,...,2,Obligations,2020,01AUG2020,75000,86804,8,Project-type interventions - not Investment Re...,30000000,60914
268943,104,MMR,Burma (Myanmar),1,East Asia and Oceania,2.0,Lower Middle Income Country,LMIC,1,USAID,...,2,Obligations,2021,01JUN2021,75000,83916,8,Project-type interventions - not Investment Re...,30000000,60914
268944,104,MMR,Burma (Myanmar),1,East Asia and Oceania,2.0,Lower Middle Income Country,LMIC,1,USAID,...,3,Disbursements,2021,01FEB2021,31126,34826,8,Project-type interventions - not Investment Re...,30000000,60914
268945,104,MMR,Burma (Myanmar),1,East Asia and Oceania,2.0,Lower Middle Income Country,LMIC,1,USAID,...,3,Disbursements,2021,01MAR2021,13033,14582,8,Project-type interventions - not Investment Re...,30000000,60914
268946,104,MMR,Burma (Myanmar),1,East Asia and Oceania,2.0,Lower Middle Income Country,LMIC,1,USAID,...,3,Disbursements,2021,01APR2021,29975,33539,8,Project-type interventions - not Investment Re...,30000000,60914
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291623,104,MMR,Burma (Myanmar),1,East Asia and Oceania,2.0,Lower Middle Income Country,LMIC,17,TDA,...,3,Disbursements,2017,14APR2017,6993,8536,13,Technical Cooperation - Other,56700,7636
291624,104,MMR,Burma (Myanmar),1,East Asia and Oceania,2.0,Lower Middle Income Country,LMIC,17,TDA,...,3,Disbursements,2017,28JUL2017,13986,17073,13,Technical Cooperation - Other,56700,7636
291625,104,MMR,Burma (Myanmar),1,East Asia and Oceania,2.0,Lower Middle Income Country,LMIC,17,TDA,...,3,Disbursements,2017,25SEP2017,6993,8536,13,Technical Cooperation - Other,56700,7636
291626,104,MMR,Burma (Myanmar),1,East Asia and Oceania,2.0,Lower Middle Income Country,LMIC,17,TDA,...,3,Disbursements,2018,01DEC2017,21978,26246,13,Technical Cooperation - Other,56700,7636


In [10]:
df_complete["Country Name"].unique()

array(['Afghanistan', 'Albania', 'Algeria', 'Angola',
       'Antigua and Barbuda', 'Azerbaijan', 'Argentina', 'Australia',
       'Austria', 'Bahamas', 'Bahrain', 'Bangladesh', 'Armenia',
       'Barbados', 'Belgium', 'Bermuda', 'Bhutan', 'Bolivia',
       'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Belize',
       'British Indian Ocean Territory', 'Solomon Islands',
       'British Virgin Islands', 'Brunei', 'Bulgaria', 'Burma (Myanmar)',
       'Burundi', 'Belarus', 'Cambodia', 'Cameroon', 'Canada',
       'Cabo Verde', 'Cayman Islands', 'Central African Republic',
       'Sri Lanka', 'Chad', 'Chile', 'China (P.R.C.)', 'Taiwan',
       'Colombia', 'Comoros', 'Congo (Brazzaville)', 'Congo (Kinshasa)',
       'Cook Islands', 'Costa Rica', 'Croatia', 'Cuba', 'Cyprus',
       'Czechoslovakia (former)', 'Czechia', 'Benin', 'Denmark',
       'Dominica', 'Dominican Republic', 'Ecuador', 'El Salvador',
       'Equatorial Guinea', 'Ethiopia', 'Eritrea', 'Estonia', 'Fiji',
       'Finlan

In [29]:
countries = [
    "Afghanistan", "Bangladesh", "Belarus", "Benin", "Brazil", "Burkina Faso", 
    "Burundi", "Central African Republic", "Chad", "Congo (Kinshasa)", "Eswatini", "Ethiopia", "Guinea", "Haiti", 
    "India", "Iraq", "Israel", "Cote d'Ivoire", "Jordan", "Kenya", "Kiribati, Republic of", "Lebanon", 
    "Liberia", "Libya", "Madagascar", "Malawi", "Malaysia", "Mali", "Mexico", 
    "Mozambique", "Burma (Myanmar)", "Nigeria", "West Bank and Gaza", "Pakistan", "Panama", 
    "Poland", "Sierra Leone", "Somalia", "South Sudan", "Sri Lanka", "Sudan", 
    "Syria", "Tajikistan", "Uganda", "Ukraine", "Uzbekistan", "Venezuela", "Yemen", 
    "Zimbabwe"
]

In [30]:
len(countries)

49

In [31]:
# Filter by relevant countries/countries of interest
df_filtered = df_complete[df_complete["Country Name"].isin(countries)]

In [32]:
# Check for differences/missing countries

# a = list(df_filtered["Country Name"].unique())
# 
# c = list(set(countries) - set(a))
# 
# print(c)

[]


In [33]:
df_filtered.head()

,Country ID,Country Code,Country Name,Region ID,Region Name,Income Group ID,Income Group Name,Income Group Acronym,Managing Agency ID,Managing Agency Acronym,...,Transaction Type ID,Transaction Type Name,Fiscal Year,Transaction Date,Current Dollar Amount,Constant Dollar Amount,aid_type_id,aid_type_name,activity_budget_amount,submission_activity_id
0,4,AFG,Afghanistan,4,South and Central Asia,1.0,Low Income Country,LIC,1,USAID,...,2,Obligations,2006,01MAR2006,37760,54932,13,Technical Cooperation - Other,.,30831
1,4,AFG,Afghanistan,4,South and Central Asia,1.0,Low Income Country,LIC,1,USAID,...,3,Disbursements,2006,01MAR2006,983,1430,13,Technical Cooperation - Other,.,30831
2,4,AFG,Afghanistan,4,South and Central Asia,1.0,Low Income Country,LIC,1,USAID,...,3,Disbursements,2006,01MAY2006,3392,4935,13,Technical Cooperation - Other,.,30831
3,4,AFG,Afghanistan,4,South and Central Asia,1.0,Low Income Country,LIC,1,USAID,...,3,Disbursements,2006,01JUL2006,5001,7275,13,Technical Cooperation - Other,.,30831
4,4,AFG,Afghanistan,4,South and Central Asia,1.0,Low Income Country,LIC,1,USAID,...,3,Disbursements,2006,01AUG2006,4257,6193,13,Technical Cooperation - Other,.,30831


In [34]:
df_filtered['Country Code'].nunique()

49

In [18]:
# Reporting on a country level (e.g., "Afghanistan")
country = "Afghanistan"

# Summarize overall funding for the country
country_summary = summarize_country_funding(df=df_filtered, country=country)
print("Country Funding Summary:")
print(country_summary)

# Get funding trends over years for the country
trends = funding_trends_by_year(df=df_filtered, country=country)
print("\nFunding Trends by Year:")
print(trends)

# Report impacts by international sector
sector_report = sector_impact_report(df=df_filtered, country=country)
print("\nSector Impact Report:")
print(sector_report)

# Simulate a full (100%) withdrawal impact for the country
withdrawal_impact = simulate_withdrawal_impact(df=df_filtered, country=country, withdrawal_fraction=1.0)
print("\nWithdrawal Impact Simulation:")
print(withdrawal_impact)

Country Funding Summary:
              num_activities  total_current_dollars  total_constant_dollars  \
Country Name                                                                  
Afghanistan            59691            54036376207             72744125786   

              avg_current_dollars  avg_constant_dollars  
Country Name                                             
Afghanistan         905268.402389          1.218678e+06  

Funding Trends by Year:
             total_current_dollars  total_constant_dollars  num_activities
Fiscal Year                                                               
1952                        300000                 2774318               1
1953                       2200000                19981696               2
1954                       2500000                22439338               1
1955                       2000000                17820438               1
1956                      18301000               159003777               3
...          